In [8]:
import os
import pandas as pd
import numpy as np
import cv2
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt

In [9]:
metadata_path = "dataset/HAM10000_metadata.csv"
df = pd.read_csv(metadata_path)
df.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [10]:
image_paths = []
image_folders = [
    "dataset/ham10000_images_part_1",
    "dataset/ham10000_images_part_2"
]
for image_id in df["image_id"]:
    image_path = None
    for folder in image_folders:
        path = os.path.join(folder,image_id + ".jpg")
        if os.path.exists(path):
            image_path = path
            break
    image_paths.append(image_path)
df["image_path"] = image_paths

In [11]:
df["image_path"].isnull().sum()

np.int64(0)

Encode the "dx" target into labelencoder()

In [12]:
encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df["dx"])

In [13]:
class_mapping = dict(zip(encoder.classes_,encoder.transform(encoder.classes_)))
class_mapping

{'akiec': np.int64(0),
 'bcc': np.int64(1),
 'bkl': np.int64(2),
 'df': np.int64(3),
 'mel': np.int64(4),
 'nv': np.int64(5),
 'vasc': np.int64(6)}

Split the dataset into train,validation and test for triain the datasets.

In [14]:
train_data, temp_data = train_test_split(df,test_size=0.30,random_state=42,stratify=df["label"])

In [15]:
val_data, test_data = train_test_split(temp_data,test_size=0.50,random_state=42,stratify=temp_data["label"])

In [16]:
print(train_data.shape,val_data.shape,test_data.shape)

(7010, 9) (1502, 9) (1503, 9)


Export the splited datasets

Image Resize and Normaliation

In [17]:
IMAGE_SIZE = 224
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    image = cv2.resize(image,(IMAGE_SIZE, IMAGE_SIZE))
    image = image / 255.0
    return image.astype(np.float32)

In [18]:
sample_image = preprocess_image(
    train_data.iloc[0]["image_path"]
)


sample_image.shape

(224, 224, 3)

In [19]:
def create_dataset(data):
    image_paths = data["image_path"].values
    labels = data["label"].values
    dataset = tf.data.Dataset.from_tensor_slices((image_paths,labels))
    def load_image(path,label):
        image = tf.numpy_function(preprocess_image,[path],tf.float32)
        image.set_shape((224,224,3))
        return image,label
    dataset = dataset.map(load_image,num_parallel_calls=tf.data.AUTOTUNE)
    return dataset

In [20]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)

In [21]:
data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
])

In [22]:
def apply_augmentation(image,label):
    image = data_augmentation(image)
    return image,label
train_dataset = train_dataset.map(apply_augmentation,num_parallel_calls=tf.data.AUTOTUNE)

In [23]:
BATCH_SIZE = 32
train_dataset = (train_dataset.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
val_dataset = (val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
test_dataset = (test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

In [24]:
class_weights = compute_class_weight(class_weight="balanced",classes=np.unique(train_data["label"]),y=train_data["label"])
class_weights

array([ 4.37305053,  2.78174603,  1.30224782, 12.3633157 ,  1.2855309 ,
        0.21338772, 10.11544012])

In [25]:
class_weights = dict(enumerate(class_weights))
class_weights

{0: np.float64(4.37305053025577),
 1: np.float64(2.7817460317460316),
 2: np.float64(1.3022478172023035),
 3: np.float64(12.36331569664903),
 4: np.float64(1.285530900421786),
 5: np.float64(0.21338772031292808),
 6: np.float64(10.115440115440116)}

In [30]:
model_results = []

In [22]:
from tensorflow.keras import layers, models

deep_cnn = models.Sequential([
    layers.Input(shape=(224,224,3)),
    # Block 1
    layers.Conv2D(32,(3,3),activation="relu"),
    layers.Conv2D(32,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    # Block 2
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    # Block 3
    layers.Conv2D(128,(3,3),activation="relu"),
    layers.Conv2D(128,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(256,activation="relu"),
    layers.Dense(7,activation="softmax")
])


deep_cnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 222, 222, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 220, 220, 32)        │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 110, 110, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 108, 108, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 106, 106, 64)        │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 53, 53, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 51, 51, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 49, 49, 128)         │         147,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 24, 24, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 73728)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │      18,874,624 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 7)                   │           1,799 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 19,163,431 (73.10 MB)

 Trainable params: 19,163,431 (73.10 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
deep_cnn.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.00001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)
early_stopping = tf.keras.callbacks.EarlyStopping(

    monitor="val_loss",

    patience=3,

    restore_best_weights=True,

    verbose=1

)

In [37]:
history_deep = deep_cnn.fit(

    train_dataset,

    validation_data=val_dataset,

    epochs=10,

    class_weight=class_weights,

    callbacks=[early_stopping]

)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 734s 3s/step - accuracy: 0.3387 - loss: 1.9214 - val_accuracy: 0.5213 - val_loss: 1.6058
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 732s 3s/step - accuracy: 0.4372 - loss: 1.8121 - val_accuracy: 0.3735 - val_loss: 1.6856
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 792s 4s/step - accuracy: 0.4361 - loss: 1.7204 - val_accuracy: 0.4700 - val_loss: 1.3892
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 680s 3s/step - accuracy: 0.4308 - loss: 1.6484 - val_accuracy: 0.4481 - val_loss: 1.3534
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 585s 3s/step - accuracy: 0.4344 - loss: 1.5752 - val_accuracy: 0.4115 - val_loss: 1.5231
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 586s 3s/step - accuracy: 0.4419 - loss: 1.4837 - val_accuracy: 0.4807 - val_loss: 1.2735
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 581s 3s/step - accuracy: 0.4690 - loss: 1.4037 - val_accuracy: 0.4754 - val_loss: 1.3043
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 657s 3s/step - accuracy: 0.4842 - loss: 1.3583 - val_accu

Optimizer sgd

In [40]:
train_loss, train_accuracy = deep_cnn.evaluate(train_dataset)
val_loss, val_accuracy = deep_cnn.evaluate(val_dataset)
test_loss, test_accuracy = deep_cnn.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 164s 709ms/step - accuracy: 0.4919 - loss: 1.2610
47/47 ━━━━━━━━━━━━━━━━━━━━ 32s 679ms/step - accuracy: 0.4807 - loss: 1.2735
47/47 ━━━━━━━━━━━━━━━━━━━━ 31s 661ms/step - accuracy: 0.4757 - loss: 1.2986


In [41]:
deep_cnn.save("C:/Users/Admin/Downloads/skin_disease/models/deep_cnn.keras") 

In [24]:
from tensorflow.keras import layers, models

deep_cnn_sgd = models.Sequential([
    layers.Input(shape=(224,224,3)),
    # Block 1
    layers.Conv2D(32,(3,3),activation="relu"),
    layers.Conv2D(32,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    # Block 2
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    # Block 3
    layers.Conv2D(128,(3,3),activation="relu"),
    layers.Conv2D(128,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(256,activation="relu"),
    layers.Dense(7,activation="softmax")
])


deep_cnn_sgd.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)                    │ (None, 222, 222, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_7 (Conv2D)                    │ (None, 220, 220, 32)        │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 110, 110, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_8 (Conv2D)                    │ (None, 108, 108, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_9 (Conv2D)                    │ (None, 106, 106, 64)        │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_4 (MaxPooling2D)       │ (None, 53, 53, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_10 (Conv2D)                   │ (None, 51, 51, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_11 (Conv2D)                   │ (None, 49, 49, 128)         │         147,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_5 (MaxPooling2D)       │ (None, 24, 24, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 73728)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 256)                 │      18,874,624 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 7)                   │           1,799 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 19,163,431 (73.10 MB)

 Trainable params: 19,163,431 (73.10 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
deep_cnn_sgd.compile(

    optimizer=tf.keras.optimizers.SGD(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_basic_sgd = deep_cnn_sgd.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=8,
    class_weight=class_weights,
    
)

Epoch 1/8


C:\Users\Admin\.virtualenvs\skin_disease-JPRYQUU3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
Exception ignored in: 'zmq.backend.cython._zmq.Frame.__dealloc__'
Traceback (most recent call last):
  File "zmq/backend/cython/_zmq.py", line 179, in zmq.backend.cython._zmq._check_rc
    PyErr_CheckSignals()
^^^^^^^^^^^
KeyboardInterrupt: 

KeyboardInterrupt



In [44]:
train_loss,train_accuracy = deep_cnn_sgd.evaluate(train_dataset)
val_loss,val_accuracy = deep_cnn_sgd.evaluate(val_dataset)
test_loss,test_accuracy = deep_cnn_sgd.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 176s 779ms/step - accuracy: 0.0484 - loss: 1.9681
47/47 ━━━━━━━━━━━━━━━━━━━━ 73s 2s/step - accuracy: 0.0559 - loss: 1.9683
47/47 ━━━━━━━━━━━━━━━━━━━━ 33s 689ms/step - accuracy: 0.0506 - loss: 1.9697


In [45]:
deep_cnn.save("C:/Users/Admin/Downloads/skin_disease/models/deep_cnn_sgd.keras") 

In [26]:
from tensorflow.keras import layers, models

deep_cnn_RMSprop = models.Sequential([
    layers.Input(shape=(224,224,3)),
    # Block 1
    layers.Conv2D(32,(3,3),activation="relu"),
    layers.Conv2D(32,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    # Block 2
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    # Block 3
    layers.Conv2D(128,(3,3),activation="relu"),
    layers.Conv2D(128,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(256,activation="relu"),
    layers.Dense(7,activation="softmax")
])


deep_cnn_RMSprop.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)                   │ (None, 222, 222, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_13 (Conv2D)                   │ (None, 220, 220, 32)        │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_6 (MaxPooling2D)       │ (None, 110, 110, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_14 (Conv2D)                   │ (None, 108, 108, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_15 (Conv2D)                   │ (None, 106, 106, 64)        │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_7 (MaxPooling2D)       │ (None, 53, 53, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_16 (Conv2D)                   │ (None, 51, 51, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_17 (Conv2D)                   │ (None, 49, 49, 128)         │         147,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_8 (MaxPooling2D)       │ (None, 24, 24, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_2 (Flatten)                  │ (None, 73728)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 256)                 │      18,874,624 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 7)                   │           1,799 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 19,163,431 (73.10 MB)

 Trainable params: 19,163,431 (73.10 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
deep_cnn_RMSprop.compile(

    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    verbose=1
)  

history_deep_cnn_RMSprop = deep_cnn_RMSprop.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 1236s 5s/step - accuracy: 0.2014 - loss: 1.9236 - val_accuracy: 0.5346 - val_loss: 1.3187
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 1068s 5s/step - accuracy: 0.4050 - loss: 1.7742 - val_accuracy: 0.3622 - val_loss: 1.6132
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 1122s 5s/step - accuracy: 0.4180 - loss: 1.6658 - val_accuracy: 0.2324 - val_loss: 1.8594
Epoch 3: early stopping
Restoring model weights from the end of the best epoch: 1.


In [28]:
train_loss,train_accuracy = deep_cnn_RMSprop.evaluate(train_dataset)
val_loss,val_accuracy = deep_cnn_RMSprop.evaluate(val_dataset)
test_loss,test_accuracy = deep_cnn_RMSprop.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 601s 3s/step - accuracy: 0.5441 - loss: 1.3230
47/47 ━━━━━━━━━━━━━━━━━━━━ 112s 2s/step - accuracy: 0.5346 - loss: 1.3187
47/47 ━━━━━━━━━━━━━━━━━━━━ 111s 2s/step - accuracy: 0.5369 - loss: 1.3469


In [29]:
deep_cnn_RMSprop.save("C:/Users/Admin/Downloads/skin_disease/models/deep_cnn_RMSprop.keras") 

In [21]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)
 
train_dataset = train_dataset.map(
    apply_augmentation,
    num_parallel_calls=tf.data.AUTOTUNE
)
 
BATCH_SIZE = 64
 
train_dataset = (
    train_dataset
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
 
val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
 
test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [22]:
from tensorflow.keras import layers, models

deep_cnn_64 = models.Sequential([
    layers.Input(shape=(224,224,3)),
    # Block 1
    layers.Conv2D(32,(3,3),activation="relu"),
    layers.Conv2D(32,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    # Block 2
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    # Block 3
    layers.Conv2D(128,(3,3),activation="relu"),
    layers.Conv2D(128,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(256,activation="relu"),
    layers.Dense(7,activation="softmax")
])


deep_cnn_64.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 220, 220, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 110, 110, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 108, 108, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 106, 106, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 53, 53, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 51, 51, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 49, 49, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 24, 24, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 73728)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    18,874,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,163,431 (73.10 MB)

 Trainable params: 19,163,431 (73.10 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
deep_cnn_64.compile(

    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

history_deep_cnn_64 = deep_cnn_64.fit(train_dataset,validation_data=val_dataset,epochs=5,

    class_weight=class_weights,callbacks=[early_stopping]
)

Epoch 1/5


c:\Users\Admin\.virtualenvs\skin_disease-JPRYQUU3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


110/110 ━━━━━━━━━━━━━━━━━━━━ 2718s 24s/step - accuracy: 0.3729 - loss: 1.8663 - val_accuracy: 0.4840 - val_loss: 1.3381
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 1867s 17s/step - accuracy: 0.4267 - loss: 1.7385 - val_accuracy: 0.4927 - val_loss: 1.2282
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 5433s 49s/step - accuracy: 0.4455 - loss: 1.5678 - val_accuracy: 0.4787 - val_loss: 1.3471
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 1510s 14s/step - accuracy: 0.4805 - loss: 1.3913 - val_accuracy: 0.5266 - val_loss: 1.2063
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 2057s 19s/step - accuracy: 0.4900 - loss: 1.3529 - val_accuracy: 0.4141 - val_loss: 1.4827
Restoring model weights from the end of the best epoch: 4.


In [24]:
train_loss,train_accuracy = deep_cnn_64.evaluate(train_dataset)
val_loss,val_accuracy = deep_cnn_64.evaluate(val_dataset)
test_loss,test_accuracy = deep_cnn_64.evaluate(test_dataset)

110/110 ━━━━━━━━━━━━━━━━━━━━ 188s 2s/step - accuracy: 0.5278 - loss: 1.1755
24/24 ━━━━━━━━━━━━━━━━━━━━ 35s 1s/step - accuracy: 0.5266 - loss: 1.2063
24/24 ━━━━━━━━━━━━━━━━━━━━ 37s 2s/step - accuracy: 0.5077 - loss: 1.2299


In [26]:
deep_cnn_64.save("C:/Users/Admin/Downloads/skin_disease/models/deep_cnn_64.keras") 

In [30]:
import keras_tuner as kt

import tensorflow as tf

from tensorflow.keras import models, layers

from tensorflow.keras.optimizers import Adam, SGD, RMSprop
 
num_classes = 7
 
def build_model(hp):
 
    final_model = models.Sequential([
 
        layers.Input(shape=(224,224,3)),
 
        layers.Conv2D(
 
            filters=hp.Choice(

                "filters_1",

                [32,64]

            ),
 
            kernel_size=(3,3),
 
            activation="relu"

        ),
 
        layers.MaxPooling2D((2,2)),
 
 
        layers.Conv2D(
 
            filters=hp.Choice(

                "filters_2",

                [64,128]

            ),
 
            kernel_size=(3,3),
 
            activation="relu"

        ),
 
        layers.MaxPooling2D((2,2)),
 
 
        layers.Flatten(),
 
 
        layers.Dense(
 
            units=hp.Choice(

                "dense_units",

                [64,128]

            ),
 
            activation="relu"

        ),
 
 
        layers.Dense(
 
            num_classes,
 
            activation="softmax"

        )
 
    ])
 
 
    learning_rate = hp.Choice(
 
        "learning_rate",
 
        [1e-2,1e-3,1e-4]

    )
 
 
    optimizer = hp.Choice(
 
        "optimizer",
 
        ["adam","sgd","rmsprop"]

    )
 
 
    if optimizer == "adam":
 
        opt = Adam(

            learning_rate=learning_rate

        )
 
    elif optimizer == "sgd":
 
        opt = SGD(

            learning_rate=learning_rate

        )
 
    else:
 
        opt = RMSprop(

            learning_rate=learning_rate

        )
 
 
    final_model.compile(
 
        optimizer=opt,
 
        loss="sparse_categorical_crossentropy",
 
        metrics=["accuracy"]

    )
 
    return final_model

 

In [33]:
 
tuner = kt.RandomSearch(
 
    build_model,
 
    objective="val_accuracy",
 
    max_trials=3,
 
    overwrite=True,
 
    directory="tuner",
 
    project_name="basic_cnn"

)


In [34]:
 
tuner.search(
 
    train_dataset,
 
    validation_data=val_dataset,
 
    epochs=5,
 
    class_weight=class_weights

)

Trial 3 Complete [00h 20m 02s]
val_accuracy: 0.6697736382484436

Best val_accuracy So Far: 0.6697736382484436
Total elapsed time: 00h 59m 46s


In [35]:
best_eff_net = tuner.get_best_models(1)[0]

/Users/aximsoft/Documents/untitled folder/.venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(store)


In [36]:
best_hps = tuner.get_best_hyperparameters(
    num_trials=1
)[0]
print(best_hps.values)

{'filters_1': 64, 'filters_2': 64, 'dense_units': 128, 'learning_rate': 0.01, 'optimizer': 'rmsprop'}


In [37]:
from tensorflow.keras import layers, models

deep_final= models.Sequential([
    layers.Input(shape=(224,224,3)),
    # Block 1
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    # Block 3
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(128,activation="relu"),
    layers.Dense(7,activation="softmax")
])


deep_final.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 222, 222, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 220, 220, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 110, 110, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 108, 108, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 106, 106, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 53, 53, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 179776)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │    23,011,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,124,935 (88.21 MB)

 Trainable params: 23,124,935 (88.21 MB)

 Non-trainable params: 0 (0.00 B)

In [38]:
deep_final.compile(

    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

history_deep_final = deep_final.fit(train_dataset,validation_data=val_dataset,epochs=5,

    class_weight=class_weights,callbacks=[early_stopping]
)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 958s 4s/step - accuracy: 0.0548 - loss: 371.1451 - val_accuracy: 0.1112 - val_loss: 1.9042
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 950s 4s/step - accuracy: 0.1805 - loss: 1.9505 - val_accuracy: 0.6698 - val_loss: 1.8965
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 972s 4s/step - accuracy: 0.2602 - loss: 1.9497 - val_accuracy: 0.1099 - val_loss: 1.9056
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 958s 4s/step - accuracy: 0.1839 - loss: 1.9490 - val_accuracy: 0.6698 - val_loss: 1.9026
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 2.


In [39]:
train_loss,train_accuracy = deep_final.evaluate(train_dataset)
val_loss,val_accuracy = deep_final.evaluate(val_dataset)
test_loss,test_accuracy = deep_final.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 215s 964ms/step - accuracy: 0.6695 - loss: 1.8966
47/47 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.6698 - loss: 1.8965
47/47 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - accuracy: 0.6693 - loss: 1.8966


In [42]:
deep_final.save("deep_final.keras") 